# Unit 2 — MLP, optimizers, SVM kernels

Phase 6. A hand-derived MLP (`docs/backprop_derivation.md`), an optimizer study and a polynomial-kernel SVM. The shipped S1 classifier is an RBF SVM on CLIP embeddings.

This notebook reads committed results from `reports/`. It reproduces nothing and needs no corpus or GPU. `reports/README.md` names the command that regenerates each file. The full syllabus mapping is in `docs/syllabus_map.md`.

In [1]:
import json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "reports").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CELLS = json.loads((ROOT / "reports" / "master_results.json").read_text(encoding="utf-8"))["cells"]


def results(*stages):
    """The master table's rows for these stages, as a Markdown table."""
    rows = ["| stage | model | metric | value | target | verdict |", "| :-- | :-- | :-- | --: | --: | :-: |"]
    for c in CELLS:
        if c["stage"] in stages and c.get("available", True):
            verdict = "" if c.get("passes") is None else ("pass" if c["passes"] else "**fail**")
            target = "" if c.get("target") is None else c["target"]
            rows.append(f"| {c['stage']} | {c['model']} | {c['metric']} | {c['value']} | {target} | {verdict} |")
    display(Markdown("\n".join(rows)))


def report(name, *keys):
    """Selected top-level fields of one reports/*.json."""
    data = json.loads((ROOT / "reports" / name).read_text(encoding="utf-8"))
    return {k: data[k] for k in keys if k in data}


## S1 — held-out-scribe accuracy

In [2]:
report("s1_heldout_scribes.json", "model", "accuracy", "accuracy_std", "macro_f1", "minimum_repeat_accuracy", "passes")

{'model': 'CLIP ViT-B/32 grayscale embedding (128 PCA components) + OvO RBF SVM',
 'accuracy': 0.9871,
 'accuracy_std': 0.0009,
 'macro_f1': 0.9518,
 'minimum_repeat_accuracy': 0.9858,
 'passes': True}

## Optimizers and schedules (6.2.4)

The learning rate matters about 20x more than the optimizer: the spread across five optimizers is 0.0173, and the spread across learning rates within SGD alone is 0.3390.

![p6_optimizers.png](../reports/figures/p6_optimizers.png)

`reports/figures/p6_optimizers.png`

![p6_schedules.png](../reports/figures/p6_schedules.png)

`reports/figures/p6_schedules.png`

## Polynomial kernel and support vectors (6.3.2)

Flowchart vs state machine is linearly separable: every cell scores 1.0000. On circuit vs flowchart, the kernel gains +0.0438.

![p6_kernel_surfaces.png](../reports/figures/p6_kernel_surfaces.png)

`reports/figures/p6_kernel_surfaces.png`

![p6_support_vectors.png](../reports/figures/p6_support_vectors.png)

`reports/figures/p6_support_vectors.png`